# Agent: Registry Control (Phase 7)

This notebook is the source of truth for criminal counts.

It subscribes to camera detection decisions and publishes official updates to:
- `simcity/surveillance/registry/updates`

Rules:
- Increment `criminal_count` only when `detected=true`
- Deduplicate by `event_id`
- Keep processing simple and explicit

In [1]:
from __future__ import annotations

import json
import threading
import time
from collections import deque
from typing import Any

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

In [2]:
# Config + MQTT contract constants
cfg = load_config()
sim_cfg = cfg.simulation

SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_REGISTRY_UPDATES = f"{SURVEILLANCE_TOPIC_ROOT}/registry/updates"

LIVE_QOS = 0
LIVE_RETAIN = False
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0

print(f"Registry broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> sub: {TOPIC_DETECTIONS}, pub: {TOPIC_REGISTRY_UPDATES}")
print(f"Registry step delay: {STEP_DELAY_S}s")

Registry broker: 127.0.0.1:1883
Topics -> sub: simcity/surveillance/detections/camera, pub: simcity/surveillance/registry/updates
Registry step delay: 1.0s


In [3]:
# Registry state + local step timing
state_lock = threading.Lock()
inbox: deque[dict[str, Any]] = deque()

anchor_step: int | None = None
anchor_started_at: float | None = None

registry: dict[str, dict[str, Any]] = {}
processed_event_ids: set[str] = set()

stats = {
    "received": 0,
    "processed": 0,
    "incremented": 0,
    "duplicates": 0,
    "dropped_invalid": 0,
    "dropped_out_of_order": 0,
}

mqtt_connector: MqttConnector | None = None
mqtt_publisher: MqttPublisher | None = None
subscription_enabled = False

def current_local_step_unlocked() -> int | None:
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)

In [4]:
# MQTT subscription callback (queue only)
def enqueue_detection(payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at

    with state_lock:
        if anchor_step is None and "step" in payload:
            try:
                anchor_step = int(payload["step"])
                anchor_started_at = time.perf_counter()
            except Exception:
                pass
        inbox.append(payload)

def on_detection_message(client, userdata, message):
    try:
        payload = json.loads(message.payload.decode("utf-8"))
        with state_lock:
            stats["received"] += 1
        enqueue_detection(payload)
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1

try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="registry-control")
    mqtt_connector.connect()

    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_publisher = MqttPublisher(mqtt_connector)
        mqtt_connector.client.on_message = on_detection_message
        mqtt_connector.client.subscribe(TOPIC_DETECTIONS, qos=LIVE_QOS)
        subscription_enabled = True
        print(f"Registry subscribed to {TOPIC_DETECTIONS}")
    else:
        print("Registry MQTT timeout. Running in idle mode.")
except Exception as exc:
    print(f"Registry MQTT unavailable ({exc}). Running in idle mode.")

Registry subscribed to simcity/surveillance/detections/camera


In [5]:
# Processing loop

def process_detection(payload: dict[str, Any]) -> None:
    required = ("step", "event_id", "human_id", "detected")
    if not all(field in payload for field in required):
        raise ValueError("Missing required detection fields")

    step = int(payload["step"])
    event_id = str(payload["event_id"])
    human_id = str(payload["human_id"])
    detected = bool(payload["detected"])

    with state_lock:
        if event_id in processed_event_ids:
            stats["duplicates"] += 1
            return
        processed_event_ids.add(event_id)

    if not detected:
        with state_lock:
            stats["processed"] += 1
        return

    with state_lock:
        record = registry.get(human_id)
        if record is None:
            record = {"name": human_id, "criminal_count": 0}
            registry[human_id] = record
        record["criminal_count"] += 1
        criminal_count = int(record["criminal_count"])
        stats["processed"] += 1
        stats["incremented"] += 1

    if mqtt_publisher is not None:
        update_payload = {
            "step": step,
            "human_id": human_id,
            "name": record["name"],
            "criminal_count": criminal_count,
        }
        mqtt_publisher.publish_json(
            TOPIC_REGISTRY_UPDATES,
            json.dumps(update_payload),
            qos=LIVE_QOS,
            retain=LIVE_RETAIN,
        )

def run_registry_loop(total_steps: int = 60) -> None:
    print("Registry loop started...")
    for _ in range(total_steps):
        step_started = time.perf_counter()

        while True:
            with state_lock:
                if not inbox:
                    break
                payload = inbox.popleft()
            try:
                process_detection(payload)
            except Exception:
                with state_lock:
                    stats["dropped_invalid"] += 1

        elapsed = time.perf_counter() - step_started
        time.sleep(max(0.0, STEP_DELAY_S - elapsed))

    print("Registry loop complete.")

In [6]:
# Start registry loop (run while detector is active)
run_registry_loop(total_steps=60)

Registry loop started...
Registry loop complete.


In [7]:
# Status snapshot
with state_lock:
    stats_snapshot = dict(stats)
    registry_snapshot = dict(registry)

print("Registry status:")
for key, value in stats_snapshot.items():
    print(f"- {key}={value}")

print("\nRegistry entries:")
if registry_snapshot:
    for human_id in sorted(registry_snapshot):
        entry = registry_snapshot[human_id]
        print(f"- {entry['name']}: criminal_count={entry['criminal_count']}")
else:
    print("- none")

Registry status:
- received=140
- processed=140
- incremented=35
- duplicates=0
- dropped_invalid=0
- dropped_out_of_order=0

Registry entries:
- Ava Jensen: criminal_count=4
- Clara Andersen: criminal_count=3
- Emma Larsen: criminal_count=6
- Freja Rasmussen: criminal_count=7
- Liam Nielsen: criminal_count=4
- Lucas Mortensen: criminal_count=1
- Noah Pedersen: criminal_count=3
- Oliver Christensen: criminal_count=3
- Sofia Madsen: criminal_count=3
- William Thomsen: criminal_count=1


In [8]:
# Cleanup
if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()
    print("Registry MQTT connector disconnected.")
else:
    print("Registry cleanup: no active MQTT connection.")

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Registry MQTT connector disconnected.
